### 提示词模板

In [ ]:
# 01 环境搭建 — 加载 .env 与 API Key
# 对应文档：docs/01_langchain环境搭建.md

import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

env_path = next(
    p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()
)
load_dotenv(env_path, override=True)

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
if os.getenv("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
if not deepseek_api_key:
    raise ValueError("没有找到 DEEPSEEK_API_KEY，请先在 .env 文件中配置。")

chat = ChatOpenAI(
    model="deepseek-chat",
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com/v1",
)

In [ ]:
# 02 PromptTemplate — 提示词模板
# 对应文档：docs/04_langchain提示词模板.md

from langchain_core.prompts import PromptTemplate

# 定义模板
template = '你是一个{role},请用{style}风格回答问题:{question}'
# 创建模板对象
prompt_template = PromptTemplate.from_template(template)

# 变量的填充
# filled_prompt = prompt_template.format(
#     role='数学老师', style='通俗易懂', question='勾股定理是什么？'
# )
filled_prompt = prompt_template.invoke({'role':'数学老师', 'style':'通俗易懂', 'question':'勾股定理是什么？'})

# 模型响应数据
ai_msg = chat.invoke(filled_prompt)
print(ai_msg.content)

In [ ]:
# 03 ChatPromptTemplate — 对话提示词模板
# 对应文档：docs/04_langchain提示词模板.md

from langchain_core.prompts import ChatPromptTemplate

# 定义模板
sys_template = "你是数学老师,请以{style}的风格回答问题"
user_template = "请用简单易懂的方式解释: {question}"

# 创建对话模板
prompt_template = ChatPromptTemplate.from_messages([
    ('system', sys_template),
    ('human', user_template)
])

# 填充变量
# prompt = prompt_template.format_messages(style='生动有趣', question='勾股定理是什么？')
prompt = prompt_template.invoke({'style':'通俗易懂', 'question':'勾股定理是什么？'})

# 获取响应
msg = chat.invoke(prompt)
print(msg.content)

### 输出格式化

In [ ]:
# 04 StructuredOutputParser — 输出格式化
# 对应文档：docs/05_输出格式化.md

from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate

# 定义输出结构
response_schemas = [
    ResponseSchema(name="name", description="人的姓名", type="string"),
    ResponseSchema(name="age", description="人的年龄", type="integer"),
]

# 创建输出解析器
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 创建提示模板，包含格式化指令
template = """你是一个信息提取助手，请从以下文本中提取姓名和年龄，并以 JSON 格式返回:
文本: {input_text}
{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)

# 填充输入
input_text = "张三今年25岁，来自北京。"
filled_prompt = prompt.format(input_text=input_text)

# 获取输出
resp = chat.invoke(filled_prompt)
parsed_output = output_parser.parse(resp.content)
print(parsed_output)

### 链式调用

In [ ]:
# 05 LCEL 链式调用 — 用 | 串联 prompt、model、parser

from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate

# 定义输出结构
response_schemas = [
    ResponseSchema(name="name", description="人的姓名", type="string"),
    ResponseSchema(name="age", description="人的年龄", type="integer"),
]

# 创建输出解析器
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 创建提示模板
template = """你是一个信息提取助手，请从以下文本中提取姓名和年龄，并以 JSON 格式返回:
文本: {input_text}
{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)

# 链式调用：prompt | model | output_parser
chain = prompt | chat | output_parser

# 一次 invoke 完成：填充变量 → 调用模型 → 解析输出
result = chain.invoke({"input_text": "张三今年25岁，来自北京。"})
print(result)